# Module Three Activity: Cleaning and Merging Weather Data
**Scenario:** Preparing weather-related data for a data infrastructure risk assessment. Three data sets are inspected, cleaned, and merged into a single structured data set that links weather measurements to station locations, supporting recommendations on how environmental factors (precipitation, temperature extremes, wind) could affect data center placement and operation.

**Data sets used:**
- `nyc_weather_2018.csv` — long-format daily weather measurements (27 measurement types) across 110 NYC-area stations for all of 2018
- `nyc_temperatures.csv` — temperature-only readings (TAVG/TMAX/TMIN) for a single station, October 2018 only
- `weather_stations.csv` — station metadata, including latitude, longitude, and elevation

In [1]:
# Import packages
import pandas as pd
import numpy as np

## Part One: Python Code
### 1. Inspect each data set

In [2]:
# Load the data sets
weather = pd.read_csv('nyc_weather_2018.csv')
temps = pd.read_csv('nyc_temperatures.csv')
stations = pd.read_csv('weather_stations.csv')

print("nyc_weather_2018.csv:", weather.shape)
weather.head()

nyc_weather_2018.csv: (78780, 5)


,date,datatype,station,attributes,value
0,2018-01-01T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,",0.0
1,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0015,",,N,",0.0
2,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,",0.0
3,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0017,",,N,",0.0
4,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0017,",,N,",0.0


In [3]:
weather.info()
print("\nMissing values:\n", weather.isna().sum())
print("\nDuplicate rows:", weather.duplicated().sum())
print("\nUnique measurement types (datatype):", weather['datatype'].nunique())
print(weather['datatype'].unique())

<class 'pandas.DataFrame'>
RangeIndex: 78780 entries, 0 to 78779
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        78780 non-null  str    
 1   datatype    78780 non-null  str    
 2   station     78780 non-null  str    
 3   attributes  78780 non-null  str    
 4   value       78780 non-null  float64
dtypes: float64(1), str(4)
memory usage: 3.0 MB

Missing values:
 date          0
datatype      0
station       0
attributes    0
value         0
dtype: int64

Duplicate rows: 0

Unique measurement types (datatype): 27
<StringArray>
['PRCP', 'SNOW', 'SNWD', 'WESF', 'WESD', 'TMAX', 'TMIN', 'TOBS', 'AWND',
 'TAVG', 'WDF2', 'WDF5', 'WSF2', 'WSF5', 'PGTM', 'DAPR', 'MDPR', 'WT01',
 'WT02', 'WT09', 'WT08', 'WT04', 'WT06', 'WT03', 'WT11', 'WT05', 'TSUN']
Length: 27, dtype: str


**Findings for `nyc_weather_2018.csv`:**
- No missing values, but the `date` column is stored as a string with a redundant `T00:00:00` time component — every value is midnight, so the time carries no information.
- The `attributes` column holds comma-separated internal QC/source-flag codes (e.g. `,,W,2400`) that aren't useful for this analysis — irrelevant column.
- `datatype` contains 27 measurement codes; many (e.g. `WT01`–`WT11` weather-type flags, `PGTM`, `WESD`) aren't relevant to the risk-assessment goal of tracking precipitation, temperature, snow, and wind.

In [4]:
print("nyc_temperatures.csv:", temps.shape)
temps.head()

nyc_temperatures.csv: (93, 5)


,date,datatype,station,attributes,value
0,2018-10-01T00:00:00,TAVG,GHCND:USW00014732,"H,,S,",21.2
1,2018-10-01T00:00:00,TMAX,GHCND:USW00014732,",,W,2400",25.6
2,2018-10-01T00:00:00,TMIN,GHCND:USW00014732,",,W,2400",18.3
3,2018-10-02T00:00:00,TAVG,GHCND:USW00014732,"H,,S,",22.7
4,2018-10-02T00:00:00,TMAX,GHCND:USW00014732,",,W,2400",26.1


In [5]:
temps.info()
print("\nMissing values:\n", temps.isna().sum())
print("\nDuplicate rows:", temps.duplicated().sum())
print("\nStations covered:", temps['station'].unique())
print("Date range:", temps['date'].min(), "to", temps['date'].max())

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        93 non-null     str    
 1   datatype    93 non-null     str    
 2   station     93 non-null     str    
 3   attributes  93 non-null     str    
 4   value       93 non-null     float64
dtypes: float64(1), str(4)
memory usage: 3.8 KB

Missing values:
 date          0
datatype      0
station       0
attributes    0
value         0
dtype: int64

Duplicate rows: 0

Stations covered: <StringArray>
['GHCND:USW00014732']
Length: 1, dtype: str
Date range: 2018-10-01T00:00:00 to 2018-10-31T00:00:00


**Findings for `nyc_temperatures.csv`:**
- Same schema and same date/attributes issues as the main weather file.
- Covers only station `GHCND:USW00014732`, only October 2018, and only `TAVG`/`TMAX`/`TMIN` — this station and these measurement types already exist inside `nyc_weather_2018.csv`, so combining the two files will create duplicate readings that need to be resolved.

In [6]:
print("weather_stations.csv:", stations.shape)
stations.head()

weather_stations.csv: (279, 5)


,id,name,latitude,longitude,elevation
0,GHCND:US1CTFR0022,"STAMFORD 2.6 SSW, CT US",41.064100,-73.577000,36.6
1,GHCND:US1CTFR0039,"STAMFORD 4.2 S, CT US",41.037788,-73.568176,6.4
2,GHCND:US1NJBG0001,"BERGENFIELD 0.3 SW, NJ US",40.921298,-74.001983,20.1
3,GHCND:US1NJBG0002,"SADDLE BROOK TWP 0.6 E, NJ US",40.902694,-74.083358,16.8
4,GHCND:US1NJBG0003,"TENAFLY 1.3 W, NJ US",40.914670,-73.977500,21.6


In [7]:
stations.info()
print("\nMissing values:\n", stations.isna().sum())
print("\nDuplicate rows:", stations.duplicated().sum())
stations[stations['elevation'].isna()]

<class 'pandas.DataFrame'>
RangeIndex: 279 entries, 0 to 278
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         279 non-null    str    
 1   name       279 non-null    str    
 2   latitude   279 non-null    float64
 3   longitude  279 non-null    float64
 4   elevation  278 non-null    float64
dtypes: float64(3), str(2)
memory usage: 11.0 KB

Missing values:
 id           0
name         0
latitude     0
longitude    0
elevation    1
dtype: int64

Duplicate rows: 0


,id,name,latitude,longitude,elevation
32,GHCND:US1NJES0020,"BLOOMFIELD 1.7 S, NJ US",40.785,-74.1885,NaN


**Findings for `weather_stations.csv`:**
- One station (`GHCND:US1NJES0020`, Bloomfield, NJ) is missing an `elevation` value.
- The key column is named `id` here but `station` in the weather files — inconsistent naming that will block a merge unless renamed.
- No duplicate rows.

### 2 & 3. Clean each data set, with rationale

In [8]:
# --- Clean nyc_weather_2018.csv ---

# Rationale: keep only measurement types relevant to infrastructure/environmental
# risk (precipitation, snow, temperature, wind). Drops administrative flag codes
# (WT01-WT11, PGTM, DAPR, MDPR, WESD, WESF, TSUN) that don't describe weather
# severity or volume.
relevant_types = ['PRCP', 'SNOW', 'SNWD', 'TMAX', 'TMIN', 'TAVG', 'TOBS', 'AWND', 'WSF2', 'WSF5']
weather_clean = weather[weather['datatype'].isin(relevant_types)].copy()

# Rationale: 'attributes' is a source/QC flag string with no analytical value
# for this task -- irrelevant column, drop it.
weather_clean = weather_clean.drop(columns=['attributes'])

# Rationale: convert date to a real datetime type; the time component is always
# 00:00:00 so it's dropped down to a date for cleaner joins/readability.
weather_clean['date'] = pd.to_datetime(weather_clean['date']).dt.date

# Rationale: rename columns for clarity and to match a consistent naming
# convention before merging with the other data sets.
weather_clean = weather_clean.rename(columns={
    'station': 'station_id',
    'datatype': 'measurement_type',
    'value': 'measurement_value'
})

# Rationale: remove exact duplicate rows, if any, to avoid double-counting
# a measurement.
before = len(weather_clean)
weather_clean = weather_clean.drop_duplicates()
print(f"Dropped {before - len(weather_clean)} duplicate rows")

weather_clean.head()

Dropped 0 duplicate rows


,date,measurement_type,station_id,measurement_value
0,2018-01-01,PRCP,GHCND:US1CTFR0039,0.0
1,2018-01-01,PRCP,GHCND:US1NJBG0015,0.0
2,2018-01-01,SNOW,GHCND:US1NJBG0015,0.0
3,2018-01-01,PRCP,GHCND:US1NJBG0017,0.0
4,2018-01-01,SNOW,GHCND:US1NJBG0017,0.0


In [9]:
# --- Clean nyc_temperatures.csv ---

# Rationale: same date/attributes cleanup and column renaming as above, so the
# schema matches nyc_weather_2018 exactly and the two can be combined.
temps_clean = temps.drop(columns=['attributes']).copy()
temps_clean['date'] = pd.to_datetime(temps_clean['date']).dt.date
temps_clean = temps_clean.rename(columns={
    'station': 'station_id',
    'datatype': 'measurement_type',
    'value': 'measurement_value'
})

temps_clean.head()

,date,measurement_type,station_id,measurement_value
0,2018-10-01,TAVG,GHCND:USW00014732,21.2
1,2018-10-01,TMAX,GHCND:USW00014732,25.6
2,2018-10-01,TMIN,GHCND:USW00014732,18.3
3,2018-10-02,TAVG,GHCND:USW00014732,22.7
4,2018-10-02,TMAX,GHCND:USW00014732,26.1


In [10]:
# --- Combine the two weather sources, then drop overlap duplicates ---

# Rationale: nyc_temperatures.csv covers a station/date/measurement combination
# that already exists in nyc_weather_2018.csv (see inspection above). Concatenate
# then drop duplicate (station_id, date, measurement_type) combinations, keeping
# the first occurrence, so each measurement is represented exactly once.
combined_weather = pd.concat([weather_clean, temps_clean], ignore_index=True)

before = len(combined_weather)
combined_weather = combined_weather.drop_duplicates(
    subset=['station_id', 'date', 'measurement_type']
)
print(f"Removed {before - len(combined_weather)} overlapping duplicate readings")
print("Combined weather shape:", combined_weather.shape)

Removed 93 overlapping duplicate readings
Combined weather shape: (67128, 4)


In [11]:
# --- Clean weather_stations.csv ---

# Rationale: rename 'id' to 'station_id' so it matches the join key used in
# the weather data -- inconsistent naming was the main issue identified above.
stations_clean = stations.rename(columns={'id': 'station_id', 'name': 'station_name'}).copy()

# Rationale: only one row (of 279) is missing elevation. Rather than drop a
# whole station's location data, fill it with the median elevation of all
# stations, since elevation is used only as a general environmental-risk
# indicator here rather than a precise measurement.
median_elev = stations_clean['elevation'].median()
stations_clean['elevation'] = stations_clean['elevation'].fillna(median_elev)

# Rationale: normalize station_name formatting (strip stray leading/trailing
# whitespace) for consistency.
stations_clean['station_name'] = stations_clean['station_name'].str.strip()

stations_clean.head()

,station_id,station_name,latitude,longitude,elevation
0,GHCND:US1CTFR0022,"STAMFORD 2.6 SSW, CT US",41.064100,-73.577000,36.6
1,GHCND:US1CTFR0039,"STAMFORD 4.2 S, CT US",41.037788,-73.568176,6.4
2,GHCND:US1NJBG0001,"BERGENFIELD 0.3 SW, NJ US",40.921298,-74.001983,20.1
3,GHCND:US1NJBG0002,"SADDLE BROOK TWP 0.6 E, NJ US",40.902694,-74.083358,16.8
4,GHCND:US1NJBG0003,"TENAFLY 1.3 W, NJ US",40.914670,-73.977500,21.6


## Part Two: Merged Data
### 4. Merge the data sets

In [12]:
# Rationale: left join keeps every weather measurement and attaches station
# location (latitude, longitude, elevation) where available. A left join
# (rather than inner) is used because we've already confirmed every station_id
# in the weather data has a matching entry in the stations data, so no rows
# should be lost -- this is validated below.
merged = combined_weather.merge(stations_clean, on='station_id', how='left')

merged.head()

,date,measurement_type,station_id,measurement_value,station_name,latitude,longitude,elevation
0,2018-01-01,PRCP,GHCND:US1CTFR0039,0.0,"STAMFORD 4.2 S, CT US",41.037788,-73.568176,6.4
1,2018-01-01,PRCP,GHCND:US1NJBG0015,0.0,"NORTH ARLINGTON 0.7 WNW, NJ US",40.791492,-74.139790,17.7
2,2018-01-01,SNOW,GHCND:US1NJBG0015,0.0,"NORTH ARLINGTON 0.7 WNW, NJ US",40.791492,-74.139790,17.7
3,2018-01-01,PRCP,GHCND:US1NJBG0017,0.0,"GLEN ROCK 0.7 SSE, NJ US",40.951090,-74.118264,28.0
4,2018-01-01,SNOW,GHCND:US1NJBG0017,0.0,"GLEN ROCK 0.7 SSE, NJ US",40.951090,-74.118264,28.0


### 5. Validate results

In [13]:
print("Merged shape:", merged.shape)
print("\nColumns:", list(merged.columns))
print("\nRemaining missing values:\n", merged.isna().sum())
print("\nDuplicate rows:", merged.duplicated().sum())
print("\nStations with no location match (should be 0):",
      merged['station_name'].isna().sum())
print("\nMeasurement types retained:", sorted(merged['measurement_type'].unique()))
print("\nDate range:", merged['date'].min(), "to", merged['date'].max())

Merged shape: (67128, 8)

Columns: ['date', 'measurement_type', 'station_id', 'measurement_value', 'station_name', 'latitude', 'longitude', 'elevation']

Remaining missing values:
 date                 0
measurement_type     0
station_id           0
measurement_value    0
station_name         0
latitude             0
longitude            0
elevation            0
dtype: int64



Duplicate rows: 0

Stations with no location match (should be 0): 0

Measurement types retained: ['AWND', 'PRCP', 'SNOW', 'SNWD', 'TAVG', 'TMAX', 'TMIN', 'TOBS', 'WSF2', 'WSF5']

Date range: 2018-01-01 to 2018-12-31


**Validation summary:** the merged data set has one row per weather measurement, with no missing station names or coordinates (confirming every station matched during the merge) and no duplicate rows. Row count reflects the deduplicated combination of both weather sources joined against the full station list, with elevation gaps filled and irrelevant columns removed.

In [14]:
# Export the merged data set for submission
merged.to_csv('merged_weather_data.csv', index=False)
print("Saved merged_weather_data.csv with", len(merged), "rows and", len(merged.columns), "columns")

Saved merged_weather_data.csv with 67128 rows and 8 columns
